# Colony: One — continue from candidate v2

The starting model won 368/600 fresh Level 4 matches against the previous candidate. This notebook starts from those verified weights and cached replay, with newer tournament examples. The saved local champion is an opponent and a final promotion gate. Training uses 160 simulations; final evaluation always uses the referee's 100.

Select A100, run the cells, and upload `level4_lab_bundle_v2.zip`. The default training budget is 45 minutes, plus final evaluation. The notebook and CUDA branch have not been executed in an attached A100 runtime here; the corresponding CPU pipeline has been tested.


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'onnx>=1.22.0', 'onnxruntime>=1.24.3', 'psutil>=7.0'])
import torch
print('Python:', sys.version.split()[0], 'PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'Select a GPU runtime first.'
print('GPU:', torch.cuda.get_device_name(0))


### Upload the isolated experiment

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded = files.upload()
archives = [name for name in uploaded if name.endswith('.zip')]
assert len(archives) == 1, 'Upload level4_lab_bundle_v2.zip.'
workspace = Path('/content/colony_level4_lab_v2')
workspace.mkdir(exist_ok=True)
with zipfile.ZipFile(archives[0]) as archive:
    for member in archive.infolist():
        target = (workspace / member.filename).resolve()
        assert target.is_relative_to(workspace.resolve()), 'Invalid archive path'
    archive.extractall(workspace)
os.chdir(workspace)
print('Workspace:', workspace)


### Save checkpoints in Drive

Only this experiment's dedicated output folder is written. Enable resume to continue the same run after a disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN = Path('/content/drive/MyDrive/colony-level4-lab/run-a100-v2')
RUN.mkdir(parents=True, exist_ok=True)
TRAIN_MINUTES = 45
GAMES_PER_ITERATION = 32
WORKERS = min(4, os.cpu_count() or 2)
FINAL_GAMES = 100
RESUME = True
SEED = 42


## Steps

### Verify rules and export compatibility before spending GPU time

In [ ]:
subprocess.check_call([sys.executable, 'checks.py', '--model', 'baselines/current.onnx'])
# Print exact dependency versions for reproducibility.
subprocess.check_call([sys.executable, '-m', 'pip', 'freeze'], stdout=(RUN / 'environment.txt').open('w'))


### Train and select a checkpoint

CPU actors run the published game and search. The parent batches their leaf evaluations on the GPU; gradients use mixed precision. Evaluation always runs the exported ONNX model on CPU, as the referee does. Iteration results are screening measurements, not final evidence. Checkpoints and replay are saved each iteration.

In [ ]:
command = [sys.executable, '-u', 'train.py', '--minutes', str(TRAIN_MINUTES),
           '--games', str(GAMES_PER_ITERATION), '--workers', str(WORKERS),
           '--simulations', '160', '--channels', '32', '--blocks', '2',
           '--warm-start', str(workspace / 'warm_start.pt'),
           '--initial-replay', str(workspace / 'initial_replay.pkl.gz'),
           '--opponent', str(workspace / 'candidates/selfplay_129_of_200.onnx'),
           '--batch-size', '256', '--eval-every', '4', '--eval-games', '16',
           '--final-games', str(FINAL_GAMES), '--final-seed', '200000000', '--seed', str(SEED), '--run', str(RUN)]
examples = workspace / 'data/tournament_training.pkl.gz'
if examples.exists():
    command += ['--examples', str(examples), '--examples-fraction', '0.25']
if RESUME:
    command.append('--resume')
subprocess.check_call(command)


## Checks

### Inspect final results and the game replays

The final holdout uses seeds excluded from checkpoint selection. More than 50% observed wins is not enough: the default promotion gate requires the lower endpoint of the 95% Wilson interval to exceed 50%. The gate must pass against both the original baseline and the archived local champion, on separate seed sets. This does not establish superiority over unknown teams. Do not repeatedly tune on these final seed sets.

In [ ]:
import json
from IPython.display import display, HTML
result = json.loads((RUN / 'final_result.json').read_text())
print(f"Wins: {result['wins']}/{result['games']} ({result['win_rate']:.1%})")
print('95% interval:', [f'{p:.1%}' for p in result['wilson_95']])
print('Promotion gate:', result['promotion_passed'])
print('Cyan wins:', result['cyan_wins'], 'Magenta wins:', result['magenta_wins'])
if 'champion_comparison' in result:
    champion = result['champion_comparison']
    print('Against local champion:', champion['wins'], '/', champion['games'])
display(HTML((RUN / 'final_match.html').read_text()))


### Validate the candidate

This runs the supplied validator on the exported file. Its full-match resource check uses the frozen Level 3 scene; the separate holdout above uses the full Level 4 scene and 100 simulations. Local acceptance cannot reproduce every server resource limit.

In [ ]:
if result['promotion_passed']:
    submission = RUN / 'submission.onnx'
    subprocess.check_call([sys.executable, 'checks.py', '--model', str(submission)])
    subprocess.check_call([sys.executable, 'engine/tools/validate_submission.py', str(submission)])
else:
    print('Improvement was not established. Retain the baseline and inspect losses.')


## Next Steps

Download only after inspecting the final results and validation. Keep the replay and checkpoint; compare against the other experiment's candidate before choosing the contest upload.

In [ ]:
if result['promotion_passed']:
    files.download(str(RUN / 'submission.onnx'))
files.download(str(RUN / 'final_match.html'))
